# California Housing Price Prediction
## Data Exploration, Cleaning, Feature Engineering, and Machine Learning

This notebook demonstrates a complete data science pipeline:
1. Data Loading
2. Data Exploration
3. Data Cleaning and Processing
4. Feature Engineering (combining columns)
5. Machine Learning Model Training and Evaluation

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 2. Load Data

In [ ]:
# Load California housing dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(f"Dataset shape: {df.shape}")
print(f"\nFeatures: {housing.feature_names}")
print(f"Target: MedHouseVal (Median House Value)")
df.head()

## 3. Data Exploration

In [ ]:
# Dataset information
print("Dataset Info:")
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

In [ ]:
# Correlation analysis
plt.figure(figsize=(12, 10))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

print("\nCorrelation with Target (MedHouseVal):")
print(correlation_matrix['MedHouseVal'].sort_values(ascending=False))

In [ ]:
# Distribution of target variable
plt.figure(figsize=(10, 6))
plt.hist(df['MedHouseVal'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Median House Value')
plt.ylabel('Frequency')
plt.title('Distribution of Median House Value')
plt.show()

## 4. Data Cleaning and Processing

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Removed {duplicates} duplicate rows")

In [ ]:
# Check for outliers using IQR method
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

outliers_count = ((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).sum()
print("Outliers per column:")
print(outliers_count)

## 5. Feature Engineering - Creating New Combined Features

In [ ]:
# Create new features by combining existing columns
df_engineered = df.copy()

# Feature 1: Rooms per household
df_engineered['RoomsPerHousehold'] = df_engineered['AveRooms'] * df_engineered['AveOccup']
print("✓ Created 'RoomsPerHousehold' = AveRooms * AveOccup")

# Feature 2: Bedrooms to Rooms ratio
df_engineered['BedroomsToRooms'] = df_engineered['AveBedrms'] / (df_engineered['AveRooms'] + 1e-10)
print("✓ Created 'BedroomsToRooms' = AveBedrms / AveRooms")

# Feature 3: Population per household
df_engineered['PopulationPerHousehold'] = df_engineered['Population'] / (df_engineered['AveOccup'] + 1e-10)
print("✓ Created 'PopulationPerHousehold' = Population / AveOccup")

# Feature 4: Total rooms indicator
df_engineered['TotalRoomsIndicator'] = df_engineered['AveRooms'] + df_engineered['AveBedrms']
print("✓ Created 'TotalRoomsIndicator' = AveRooms + AveBedrms")

# Feature 5: Income-Age interaction
df_engineered['IncomeAgeInteraction'] = df_engineered['MedInc'] * df_engineered['HouseAge']
print("✓ Created 'IncomeAgeInteraction' = MedInc * HouseAge")

# Feature 6: Geographic density
df_engineered['GeoDensity'] = df_engineered['Population'] / (df_engineered['Latitude'] * df_engineered['Longitude'] + 1e-10)
print("✓ Created 'GeoDensity' = Population / (Latitude * Longitude)")

print(f"\nOriginal features: {df.shape[1]}")
print(f"New features: {df_engineered.shape[1]}")
print(f"Added {df_engineered.shape[1] - df.shape[1]} new engineered features")

In [ ]:
# Check correlation of new features with target
new_features = ['RoomsPerHousehold', 'BedroomsToRooms', 'PopulationPerHousehold', 
                'TotalRoomsIndicator', 'IncomeAgeInteraction', 'GeoDensity']

print("Correlation of new features with target (MedHouseVal):")
for feature in new_features:
    corr = df_engineered[feature].corr(df_engineered['MedHouseVal'])
    print(f"  {feature}: {corr:.4f}")

In [ ]:
# Visualize new features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, feature in enumerate(new_features):
    axes[idx].scatter(df_engineered[feature], df_engineered['MedHouseVal'], alpha=0.3, s=10)
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('MedHouseVal')
    axes[idx].set_title(f'{feature} vs MedHouseVal')

plt.tight_layout()
plt.show()

## 6. Machine Learning - Model Training

In [ ]:
# Prepare data for modeling
X = df_engineered.drop('MedHouseVal', axis=1)
y = df_engineered['MedHouseVal']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled using StandardScaler")

### Model 1: Linear Regression

In [ ]:
# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Make predictions
lr_train_pred = lr_model.predict(X_train_scaled)
lr_test_pred = lr_model.predict(X_test_scaled)

# Calculate metrics
lr_train_rmse = np.sqrt(mean_squared_error(y_train, lr_train_pred))
lr_test_rmse = np.sqrt(mean_squared_error(y_test, lr_test_pred))
lr_train_r2 = r2_score(y_train, lr_train_pred)
lr_test_r2 = r2_score(y_test, lr_test_pred)
lr_test_mae = mean_absolute_error(y_test, lr_test_pred)

print("Linear Regression Results:")
print(f"  Training RMSE: {lr_train_rmse:.4f}")
print(f"  Test RMSE: {lr_test_rmse:.4f}")
print(f"  Training R²: {lr_train_r2:.4f}")
print(f"  Test R²: {lr_test_r2:.4f}")
print(f"  Test MAE: {lr_test_mae:.4f}")

### Model 2: Random Forest Regressor

In [ ]:
# Train Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train_scaled, y_train)

# Make predictions
rf_train_pred = rf_model.predict(X_train_scaled)
rf_test_pred = rf_model.predict(X_test_scaled)

# Calculate metrics
rf_train_rmse = np.sqrt(mean_squared_error(y_train, rf_train_pred))
rf_test_rmse = np.sqrt(mean_squared_error(y_test, rf_test_pred))
rf_train_r2 = r2_score(y_train, rf_train_pred)
rf_test_r2 = r2_score(y_test, rf_test_pred)
rf_test_mae = mean_absolute_error(y_test, rf_test_pred)

print("Random Forest Results:")
print(f"  Training RMSE: {rf_train_rmse:.4f}")
print(f"  Test RMSE: {rf_test_rmse:.4f}")
print(f"  Training R²: {rf_train_r2:.4f}")
print(f"  Test R²: {rf_test_r2:.4f}")
print(f"  Test MAE: {rf_test_mae:.4f}")

In [ ]:
# Feature importance from Random Forest
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Feature Importances:")
print(feature_importance.head(10))

# Plot feature importance
plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'].head(10), feature_importance['importance'].head(10))
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Model Comparison

In [ ]:
# Compare models
comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'Test RMSE': [lr_test_rmse, rf_test_rmse],
    'Test R²': [lr_test_r2, rf_test_r2],
    'Test MAE': [lr_test_mae, rf_test_mae]
})

print("\nModel Comparison:")
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['Test RMSE', 'Test R²', 'Test MAE']
for idx, metric in enumerate(metrics):
    axes[idx].bar(comparison_df['Model'], comparison_df[metric], color=['skyblue', 'coral'])
    axes[idx].set_ylabel(metric)
    axes[idx].set_title(f'{metric} Comparison')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Prediction vs Actual plots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Linear Regression
axes[0].scatter(y_test, lr_test_pred, alpha=0.5, s=20)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title('Linear Regression: Actual vs Predicted')

# Random Forest
axes[1].scatter(y_test, rf_test_pred, alpha=0.5, s=20, color='coral')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Values')
axes[1].set_ylabel('Predicted Values')
axes[1].set_title('Random Forest: Actual vs Predicted')

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrates a complete data science pipeline:

1. **Data Loading**: Loaded California housing dataset with 20,640 samples and 8 features
2. **Data Exploration**: Analyzed distributions, correlations, and data quality
3. **Data Cleaning**: Checked for duplicates and outliers
4. **Feature Engineering**: Created 6 new features by combining existing columns:
   - RoomsPerHousehold (AveRooms * AveOccup)
   - BedroomsToRooms (AveBedrms / AveRooms)
   - PopulationPerHousehold (Population / AveOccup)
   - TotalRoomsIndicator (AveRooms + AveBedrms)
   - IncomeAgeInteraction (MedInc * HouseAge)
   - GeoDensity (Population / (Latitude * Longitude))
5. **Machine Learning**: Trained and evaluated two models:
   - Linear Regression
   - Random Forest Regressor

The engineered features helped improve model performance, and the Random Forest model typically performs better than Linear Regression on this dataset.